# 5. Análise dos resultados e exportação para o Power BI
Este notebook resume a classificação pelos eixos da BNCC e organiza os dados em tabelas que podem ser importadas no Power BI.

In [2]:
from pathlib import Path
import pandas as pd
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.worksheet.table import Table, TableStyleInfo

inicio = Path.cwd().resolve()
raiz = next((p for p in (inicio, *inicio.parents) if (p / 'README.md').exists() and (p / 'dados').exists()), None)
if raiz is None:
    raise FileNotFoundError('Não foi possível localizar a pasta do projeto.')

pasta_processados = raiz / 'dados' / '1_processados'
pasta_consumo = raiz / 'dados' / '2_consumo'
pasta_consumo.mkdir(parents=True, exist_ok=True)

## 5.1 Leitura dos resultados da classificação

In [3]:
artigos = pd.read_csv(pasta_processados / '04_artigos_classificados_bncc.csv', encoding='utf-8-sig')
classificacoes = pd.read_csv(pasta_processados / '04_classificacoes_bncc.csv', encoding='utf-8-sig')
frequencia_descritores = pd.read_csv(pasta_processados / '04_frequencia_descritores_bncc.csv', encoding='utf-8-sig')
termos_titulos = pd.read_csv(pasta_processados / '03_termos_titulos.csv', encoding='utf-8-sig')
bigramas_titulos = pd.read_csv(pasta_processados / '03_bigramas_titulos.csv', encoding='utf-8-sig')
ranking_termos = pd.read_csv(pasta_processados / '03_ranking_termos_titulos.csv', encoding='utf-8-sig')
ranking_bigramas = pd.read_csv(pasta_processados / '03_ranking_bigramas_titulos.csv', encoding='utf-8-sig')
freq_termos_ano_evento = pd.read_csv(pasta_processados / '03_frequencia_termos_ano_evento.csv', encoding='utf-8-sig')
freq_bigramas_ano_evento = pd.read_csv(pasta_processados / '03_frequencia_bigramas_ano_evento.csv', encoding='utf-8-sig')

artigos['ano'] = pd.to_numeric(artigos['ano'], errors='coerce').astype('Int64')
classificacoes['ano'] = pd.to_numeric(classificacoes['ano'], errors='coerce').astype('Int64')
print(f'Artigos analisados: {len(artigos)}')
print(f'Relações artigo–eixo: {len(classificacoes)}')

Artigos analisados: 2108
Relações artigo–eixo: 3254


## 5.2 Indicadores gerais

In [4]:
total_artigos = len(artigos)
total_classificados = int((artigos['quantidade_eixos'] > 0).sum())
total_nao_classificados = total_artigos - total_classificados
cobertura = round(total_classificados / total_artigos * 100, 2)

indicadores = pd.DataFrame({
    'indicador': ['Artigos analisados', 'Artigos classificados', 'Artigos não classificados', 'Cobertura da classificação (%)'],
    'valor': [total_artigos, total_classificados, total_nao_classificados, cobertura],
})
indicadores

,indicador,valor
0,Artigos analisados,2108.00
1,Artigos classificados,1903.00
2,Artigos não classificados,205.00
3,Cobertura da classificação (%),90.28


## 5.3 Distribuição por eixo

In [5]:
resumo_eixos = (classificacoes.groupby('eixo_bncc')['id_artigo']
    .nunique().rename('quantidade_artigos').reset_index())
resumo_eixos['percentual_corpus'] = (resumo_eixos['quantidade_artigos'] / total_artigos * 100).round(2)
resumo_eixos = resumo_eixos.sort_values('quantidade_artigos', ascending=False)
resumo_eixos

,eixo_bncc,quantidade_artigos,percentual_corpus
2,Pensamento Computacional,1240,58.82
0,Cultura Digital,1097,52.04
1,Mundo Digital,917,43.50


In [6]:
resumo_evento_eixo = (classificacoes.groupby(['evento', 'eixo_bncc'])['id_artigo']
    .nunique().rename('quantidade_artigos').reset_index())
resumo_evento_eixo.pivot(index='eixo_bncc', columns='evento', values='quantidade_artigos').fillna(0)

evento,WEI,WIE
eixo_bncc,,
Cultura Digital,305,792
Mundo Digital,347,570
Pensamento Computacional,668,572


## 5.4 Evolução anual e cobertura

In [7]:
resumo_ano_evento = (artigos.assign(classificado=artigos['quantidade_eixos'].gt(0))
    .groupby(['ano', 'evento'])
    .agg(total_artigos=('id_artigo', 'nunique'), artigos_classificados=('classificado', 'sum'))
    .reset_index())
resumo_ano_evento['artigos_nao_classificados'] = resumo_ano_evento['total_artigos'] - resumo_ano_evento['artigos_classificados']
resumo_ano_evento['cobertura_percentual'] = (resumo_ano_evento['artigos_classificados'] / resumo_ano_evento['total_artigos'] * 100).round(2)
resumo_ano_evento

,ano,evento,total_artigos,artigos_classificados,artigos_nao_classificados,cobertura_percentual
0,2007,WEI,16,16,0,100.00
1,2008,WEI,30,28,2,93.33
2,2009,WEI,27,23,4,85.19
3,2010,WEI,25,24,1,96.00
4,2010,WIE,60,46,14,76.67
5,2011,WEI,26,25,1,96.15
6,2011,WIE,71,65,6,91.55
7,2012,WEI,37,37,0,100.00
8,2012,WIE,48,43,5,89.58
9,2013,WEI,51,48,3,94.12


In [8]:
resumo_ano_evento_eixo = (classificacoes.groupby(['ano', 'evento', 'eixo_bncc'])['id_artigo']
    .nunique().rename('quantidade_artigos').reset_index())
resumo_ano_evento_eixo.head(12)

,ano,evento,eixo_bncc,quantidade_artigos
0,2007,WEI,Cultura Digital,4
1,2007,WEI,Mundo Digital,11
2,2007,WEI,Pensamento Computacional,11
3,2008,WEI,Cultura Digital,8
4,2008,WEI,Mundo Digital,13
5,2008,WEI,Pensamento Computacional,26
6,2009,WEI,Cultura Digital,8
7,2009,WEI,Mundo Digital,13
8,2009,WEI,Pensamento Computacional,19
9,2010,WEI,Cultura Digital,7


## 5.5 Sobreposição entre os eixos

In [9]:
sobreposicao = (artigos['quantidade_eixos'].value_counts().sort_index()
    .rename_axis('quantidade_eixos').reset_index(name='quantidade_artigos'))
sobreposicao['percentual_corpus'] = (sobreposicao['quantidade_artigos'] / total_artigos * 100).round(2)
sobreposicao

,quantidade_eixos,quantidade_artigos,percentual_corpus
0,0,205,9.72
1,1,785,37.24
2,2,885,41.98
3,3,233,11.05


## 5.6 Preparação do modelo de dados

In [10]:
mapa_eixos = {
    'Pensamento Computacional': 'PC',
    'Mundo Digital': 'MD',
    'Cultura Digital': 'CD',
}

fato_artigos = artigos[['id_artigo', 'evento', 'ano', 'titulo', 'resumo', 'palavras_chave', 'url', 'quantidade_eixos']].copy()
fato_artigos['classificado_bncc'] = fato_artigos['quantidade_eixos'].gt(0)

ponte_artigo_eixo = classificacoes.rename(columns={
    'quantidade_evidencias': 'qtd_evidencias',
    'termos_encontrados': 'evidencias',
}).copy()
ponte_artigo_eixo.insert(1, 'id_eixo', ponte_artigo_eixo['eixo_bncc'].map(mapa_eixos))

dim_eixos = pd.DataFrame([
    {'id_eixo': 'PC', 'eixo_bncc': 'Pensamento Computacional', 'ordem': 1},
    {'id_eixo': 'MD', 'eixo_bncc': 'Mundo Digital', 'ordem': 2},
    {'id_eixo': 'CD', 'eixo_bncc': 'Cultura Digital', 'ordem': 3},
])
dim_eventos = pd.DataFrame({'evento': sorted(artigos['evento'].dropna().unique())})
dim_anos = pd.DataFrame({'ano': sorted(artigos['ano'].dropna().astype(int).unique())})

tabela_artigo_bncc = classificacoes[['id_artigo', 'eixo_bncc']].copy()
tabela_artigo_bncc['eixo_bncc'] = tabela_artigo_bncc['eixo_bncc'].astype(str).str.strip()
tabela_artigo_bncc = (
    tabela_artigo_bncc[tabela_artigo_bncc['eixo_bncc'] != '']
    .drop_duplicates()
    .reset_index(drop=True)
)

freq_bncc_ano_evento = (
    tabela_artigo_bncc
    .merge(artigos[['id_artigo', 'ano', 'evento']], on='id_artigo', how='left')
    .groupby(['ano', 'evento', 'eixo_bncc'])
    .size()
    .reset_index(name='frequencia')
)

print(f'Artigos: {len(fato_artigos)}')
print(f'Relações artigo–eixo: {len(ponte_artigo_eixo)}')
print(f'Linhas tabela BNCC: {len(tabela_artigo_bncc)}')
print(f'Frequências BNCC por ano/evento: {len(freq_bncc_ano_evento)}')

Artigos: 2108
Relações artigo–eixo: 3254
Linhas tabela BNCC: 3254
Frequências BNCC por ano/evento: 105


## 5.7 Exportação dos arquivos XLSX

In [ ]:
def formatar_workbook(writer):
    for indice, planilha in enumerate(writer.book.worksheets, start=1):
        planilha.freeze_panes = 'A2'
        planilha.auto_filter.ref = planilha.dimensions
        for celula in planilha[1]:
            celula.fill = PatternFill('solid', fgColor='1F4E78')
            celula.font = Font(color='FFFFFF', bold=True)
            celula.alignment = Alignment(horizontal='center')
        for coluna in planilha.columns:
            valores = [str(c.value) if c.value is not None else '' for c in coluna[:200]]
            largura = min(max(max((len(v) for v in valores), default=0) + 2, 12), 60)
            planilha.column_dimensions[coluna[0].column_letter].width = largura
        if planilha.max_row > 1 and planilha.max_column > 0:
            tabela = Table(displayName=f'Tabela{indice}', ref=planilha.dimensions)
            tabela.tableStyleInfo = TableStyleInfo(name='TableStyleMedium2', showRowStripes=True)
            planilha.add_table(tabela)


def criar_identificador_unico(df: pd.DataFrame, coluna: str) -> pd.DataFrame:
    df = df.copy()
    if 'evento' in df.columns and coluna in df.columns:
        df[f'{coluna}_original'] = df[coluna].astype(str).str.strip()
        df[coluna] = (
            df['evento'].fillna('').astype(str).str.strip()
            + ' | '
            + df[coluna].fillna('').astype(str).str.strip()
        )
    return df

# Garante que termos e bigramas fiquem com identificadores únicos para o Power BI
termos_titulos = criar_identificador_unico(termos_titulos, 'termo')
bigramas_titulos = criar_identificador_unico(bigramas_titulos, 'bigrama')
ranking_termos = criar_identificador_unico(ranking_termos, 'termo')
ranking_bigramas = criar_identificador_unico(ranking_bigramas, 'bigrama')
freq_termos_ano_evento = criar_identificador_unico(freq_termos_ano_evento, 'termo')
freq_bigramas_ano_evento = criar_identificador_unico(freq_bigramas_ano_evento, 'bigrama')

arquivo_powerbi = pasta_consumo / '05_powerbi_bncc.xlsx'
with pd.ExcelWriter(arquivo_powerbi, engine='openpyxl') as writer:
    artigos.to_excel(writer, sheet_name='Artigos', index=False)
    termos_titulos.to_excel(writer, sheet_name='Termos', index=False)
    bigramas_titulos.to_excel(writer, sheet_name='Bigramas', index=False)
    ranking_termos.to_excel(writer, sheet_name='Ranking_Termos_Relevantes', index=False)
    ranking_bigramas.to_excel(writer, sheet_name='Ranking_Bigramas_Relevantes', index=False)
    freq_termos_ano_evento.to_excel(writer, sheet_name='Freq_Termos_Ano_Evento', index=False)
    freq_bigramas_ano_evento.to_excel(writer, sheet_name='Freq_Bigramas_Ano_Evento', index=False)
    tabela_artigo_bncc.to_excel(writer, sheet_name='Artigo_BNCC', index=False)
    freq_bncc_ano_evento.to_excel(writer, sheet_name='Freq_BNCC_Ano_Evento', index=False)
    formatar_workbook(writer)

print(f'Gerado: {arquivo_powerbi}')

Gerado: C:\Users\laiss\OneDrive\Documentos\TCC_2026\tcc_eng_dados\dados\2_consumo\05_powerbi_bncc.xlsx
